# Praktikum Keamanan Siber Berbasis Sistem Cerdas
## Topik: Dari Deteksi Statis Menuju Pertahanan Adaptif (Peran AI dalam Keamanan Siber)
### Studi Kasus: Network Intrusion Detection System (NIDS) Menggunakan Random Forest Classifier

---
### 📌 Latar Belakang & Masalah
1. **Ancaman Siber Bersifat Dinamis:** Serangan siber modern (seperti DoS SYN Flood dan Port Scanning siluman) terus berevolusi dengan cepat.
2. **Keterbatasan Keamanan Tradisional:** Metode lama berbasis aturan (*rule-based*) atau *signature-based* (seperti antivirus/firewall konvensional) bersifat statis dan **gagal mendeteksi ancaman baru / varian zero-day**.
3. **Kebutuhan Sistem Cerdas Adaptif:** Kita memerlukan Machine Learning yang dapat belajar dari data perilaku lalu lintas jaringan (*behavior-based*) dan merespons ancaman secara *real-time*.


---
## 🏛️ Pilar #1: Data adalah Bahan Bakar
Data mentah yang digunakan bersumber dari **Network Traffic (NetFlow/PCAP Logs)**.
Fitur-fitur mencakup durasi koneksi, protokol (TCP/UDP/ICMP), jenis layanan (HTTP, DNS, SSH, dll.), status TCP flag, jumlah bytes, dan frekuensi koneksi.


In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# 1. Load Dataset
df = pd.read_csv('network_traffic_dataset.csv')
print(f'Dimensi Dataset: {df.shape[0]} baris x {df.shape[1]} kolom')
df.head()


ModuleNotFoundError: No module named 'pandas'

---
## 📊 Exploratory Data Analysis (EDA)
Melihat distribusi kelas target untuk mendeteksi apakah data mengalami **Class Imbalance**.
Di dunia nyata, lalu lintas normal berjumlah jauh lebih banyak daripada serangan (serangan adalah anomali langka).


In [ ]:
# Distribusi Kelas Target
print(df['label'].value_counts())
print('\nProporsi Kelas:')
print(df['label'].value_counts(normalize=True) * 100)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
df['label'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c', '#e67e22'])
plt.title('Distribusi Kelas (Imbalanced Data)')
plt.ylabel('Jumlah Paket/Koneksi')

plt.subplot(1, 2, 2)
df['label'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c', '#e67e22'])
plt.title('Proporsi Normal vs Serangan')
plt.show()


NameError: name 'df' is not defined

---
## ⚙️ Pilar #2: Preprocessing Data (Garbage In, Garbage Out)
Tahapan penting preprocessing:
1. **Data Cleaning:** Menghapus missing values dan duplikasi.
2. **Feature Engineering:** Mengekstrak atribut domain keamanan:
   - yte_ratio: Rasio pengiriman data dari klien ke server (mengidentifikasi flooding searah).
   - srv_concentration: Rasio koneksi ke port target tertentu.
3. **Encoding:** Mengubah fitur kategorikal (protocol_type, service, 
lag) menjadi representasi numerik dengan One-Hot Encoding.
4. **Normalisasi / Standarisasi:** Menggunakan StandardScaler (Z-score) agar skala bytes tidak mendominasi fitur rasio.
5. **Menangani Imbalanced Data:** Menggunakan **SMOTE (Synthetic Minority Over-sampling Technique)** pada data training.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# 1. Data Cleaning
print('Missing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())

# 2. Feature Engineering
df['byte_ratio'] = df['src_bytes'] / (df['dst_bytes'] + 1.0)
df['srv_concentration'] = df['srv_count'] / (df['count'] + 1.0)

# 3. Encoding Fitur Kategorikal
X = df.drop(columns=['label'])
y = df['label']

cat_cols = ['protocol_type', 'service', 'flag']
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
feature_names = X_encoded.columns.tolist()

# 4. Train-Test Split (80% Train, 20% Test) dengan Stratify
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Normalisasi / Standarisasi
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Menangani Imbalanced Data dengan SMOTE (HANYA PADA DATA TRAIN!)
print('Distribusi Train SEBELUM SMOTE:\n', y_train.value_counts())
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print('\nDistribusi Train SESUDAH SMOTE:\n', y_train_resampled.value_counts())


ModuleNotFoundError: No module named 'sklearn'

---
## 🤖 Pilar #3: Model AI (Mesin Deteksi - Supervised Learning)
### Algoritma: Random Forest Classifier
**Mengapa Random Forest?**
- **Ensemble Bagging:** Menggabungkan 100 Decision Tree independen untuk menghasilkan prediksi yang stabil dan tahan terhadap overfitting/noise.
- **Explainability (Feature Importance):** Memberikan transparansi kepada analis SOC (*Security Operations Center*) mengenai faktor utama kenapa suatu paket diblokir.
- **Kemampuan Non-Linear:** Sangat andal memisahkan pola lalu lintas normal dari anomali serangan.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=1
)

rf_model.fit(X_train_resampled, y_train_resampled)
print('Training Random Forest Selesai!')


ModuleNotFoundError: No module named 'sklearn'

---
## 🧪 Evaluasi & Testing
### Analisis Kritis Keamanan Siber: False Positive vs False Negative
- **False Positive (FP):** Traffic normal dicurigai sebagai serangan -> Menyebabkan *alert fatigue* pada analis SOC.
- **False Negative (FN):** Serangan siber lolos dan dianggap normal -> **BENCANA BESAR!** Server bisa lumpuh atau data dicuri.
- Oleh karena itu, metrik **Recall** untuk kelas serangan (DoS_SYN_Flood dan Port_Scan) harus setinggi mungkin (mendekati 1.00).


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = rf_model.predict(X_test_scaled)
target_classes = sorted(list(y_test.unique()))

print(f'Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n')
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, digits=4))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred, labels=target_classes)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_classes, yticklabels=target_classes)
plt.title('Confusion Matrix - NIDS Model')
plt.xlabel('Prediksi AI')
plt.ylabel('Kondisi Nyata')
plt.show()


ModuleNotFoundError: No module named 'sklearn'

---
## 🔍 Explainable AI (XAI): Feature Importance
Melihat atribut jaringan apa yang paling menentukan deteksi serangan:


In [ ]:
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='importance', y='feature', data=feat_df.head(8), palette='viridis')
plt.title('Top Feature Importance (Indikator Serangan Terkuat)')
plt.show()
feat_df.head(8)


NameError: name 'rf_model' is not defined

---
## 🛡️ Inference & Deploy: Menuju Pertahanan Adaptif Real-Time
Model yang telah dilatih digunakan untuk memproses log paket lalu lintas baru secara real-time dan secara otomatis mengeksekusi mitigasi keamanan (misal blokir IP pada firewall).


In [ ]:
import inference_deploy

# Menjalankan simulasi respon adaptif
inference_deploy.run_inference_simulation()


ModuleNotFoundError: No module named 'joblib'